# AI Biomedical Data Preprocessing Agent
## Interactive Evaluation & Verification Notebook

This notebook demonstrates the execution of the **AI-Powered Biomedical Data Preprocessing and Feature Engineering Agent** on the **Survey Lung Cancer Dataset** (data/input/lung_cancer.csv).

### Core Capabilities Demonstrated:
1. **Schema & Quality Inspection**: Deterministic profiling and candidate target ranking.
2. **Dual-Engine Preprocessing Plan**: Automated planning targeting 4 features for future 4-Qubit VQC.
3. **Leakage-Free Execution**: Strict train-fit, validation/test-transform isolation.
4. **Validation & Verification**: Multi-condition dataset validation.
5. **Inference Pipeline Serialization**: Testing itted_pipeline.joblib on simulated patient inference records.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Add project root to path
agent_root = Path('..').resolve()
if str(agent_root) not in sys.path:
    sys.path.insert(0, str(agent_root))

from app.agent.preprocessing_agent import PreprocessingAgent
from app.pipeline.preprocessing_pipeline import PreprocessingPipeline
print('PreprocessingAgent imports successfully resolved.')


### 1. Initialize Preprocessing Agent with Lung Cancer Configuration

In [ ]:
agent = PreprocessingAgent(config_path='../config/lung_cancer.yaml')
print('Agent initialized with config: ../config/lung_cancer.yaml')


### 2. Execute Full Preprocessing Workflow

In [ ]:
result = agent.run(
    dataset_path='../data/input/lung_cancer.csv',
    mode='auto',
    output_dir='../data/output'
)

print(f'Run ID:            {result.run_id}')
print(f'Status:            {result.status.upper()}')
print(f'Target Column:     {result.target_column}')
print(f'Task Type:         {result.task_type}')
print(f'Selected Features: {result.feature_selection.selected_features}')
print(f'Validation:        {result.validation.status.upper()}')


### 3. Inspect Processed Dataset Splits

In [ ]:
X_train = pd.read_csv('../data/output/X_train.csv')
X_val = pd.read_csv('../data/output/X_validation.csv')
X_test = pd.read_csv('../data/output/X_test.csv')
y_train = pd.read_csv('../data/output/y_train.csv')

print(f'X_train shape: {X_train.shape}')
print(f'X_val shape:   {X_val.shape}')
print(f'X_test shape:  {X_test.shape}')
print('\nFirst 5 rows of X_train:')
display(X_train.head())


### 4. Verify Feature Ranking and Quantum 4-Feature Budget

In [ ]:
ranking_df = pd.DataFrame([f.model_dump() for f in result.feature_selection.ranking])
ranking_df['Selected for VQC'] = ranking_df['feature_name'].isin(result.feature_selection.selected_features)
display(ranking_df)


### 5. Verify Serialized Pipeline on Unseen Patient Records

In [ ]:
# Load serialized inference pipeline
pipeline = PreprocessingPipeline.load('../data/output/fitted_pipeline.joblib')

# Simulated new patient records
new_patients = pd.DataFrame({
    'GENDER': ['M', 'F'],
    'AGE': [62, 58],
    'SMOKING': [2, 1],
    'YELLOW_FINGERS': [2, 1],
    'ANXIETY': [1, 2],
    'PEER_PRESSURE': [2, 1],
    'CHRONIC_DISEASE': [1, 2],
    'FATIGUE': [2, 1],
    'ALLERGY': [1, 2],
    'WHEEZING': [2, 1],
    'ALCOHOL_CONSUMING': [2, 1],
    'COUGHING': [2, 1],
    'SHORTNESS_OF_BREATH': [2, 1],
    'SWALLOWING_DIFFICULTY': [2, 1],
    'CHEST_PAIN': [2, 1]
})

transformed_features = pipeline.transform(new_patients)
print('Transformed Features for Quantum VQC Input (4 features in [0, 1]):')
display(transformed_features)
